### 0) Comentario de prueba (Primer commit)

In [1]:
print("Comentario para crear un commit en el código")

Comentario para crear un commit en el código


### 1) Cargar datos
Carga del archivo CSV, se muestran sus dimensiones y primeras filas.

In [3]:

import pandas as pd
import numpy as np


pd.set_option('display.max_columns', None)

# Ajusta la ruta si es necesario
df = pd.read_csv('BikePrices.CSV')

print('Shape:', df.shape)
df.head(3)



Shape: (1061, 8)


,Brand,Model,Selling_Price,Year,Seller_Type,Owner,KM_Driven,Ex_Showroom_Price
0,TVS,TVS XL 100,30000,2017,Individual,1st owner,8000,30490.0
1,Bajaj,Bajaj ct 100,18000,2017,Individual,1st owner,35000,32000.0
2,Yo,Yo Style,20000,2011,Individual,1st owner,10000,37675.0


### 2) Vista general y tipos de datos
Se revisan los tipos de datos y columnas por tipo.

In [9]:

print('\nTipos por frecuencia:')
print(df.dtypes.value_counts())
list(df.columns)



Tipos por frecuencia:
object     4
int64      3
float64    1
Name: count, dtype: int64


['brand',
 'model',
 'selling_price',
 'year',
 'seller_type',
 'owner',
 'km_driven',
 'ex_showroom_price']

### 3) Estandarizar nombres de columnas
Se limpian espacios en blanco antes y después de los nombres

In [8]:

# Quitar únicamente espacios iniciales y finales en los nombres de columnas
df.columns = [c.strip() for c in df.columns]
df.head(1)



,brand,model,selling_price,year,seller_type,owner,km_driven,ex_showroom_price
0,TVS,TVS XL 100,30000,2017,Individual,1st owner,8000,30490.0


### 4) Limpieza de texto y datos
Se eliminamn espacios en blanco y normalizan strings.

In [11]:

obj_cols = df.select_dtypes(include=['object']).columns
for c in obj_cols:
    df[c] = df[c].astype(str).str.strip().replace({'': np.nan})
df.head(1)


,brand,model,selling_price,year,seller_type,owner,km_driven,ex_showroom_price
0,TVS,TVS XL 100,30000,2017,Individual,1st owner,8000,30490.0


### 5) Conversión de tipos
 Conversión a numérico/fecha cuando es posible.

In [15]:

# Intento masivo a numérico para columnas 'object' (sin forzar)
for c in obj_cols:
    cn = pd.to_numeric(df[c], errors='ignore')
    if not isinstance(cn, pd.Series) or cn.dtype == 'O':
        continue
    df[c] = cn

df.head(1)


C:\Users\USUARIO\AppData\Local\Temp\ipykernel_25000\850336458.py:3: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  cn = pd.to_numeric(df[c], errors='ignore')


,brand,model,selling_price,year,seller_type,owner,km_driven,ex_showroom_price
0,TVS,TVS XL 100,30000,2017,Individual,1st owner,8000,30490.0


### 6) Resumen estadístico y cardinalidad
Calcula estadísticos básicos y cantidad de valores únicos por columna.

In [19]:
# Convertir fechas temporalmente a numérico para el describe
df_temp = df.copy()
for col in df_temp.select_dtypes(include=['datetime']):
    df_temp[col] = df_temp[col].astype('int64')  # nanosegundos desde epoch

# Resumen estadístico
display(df_temp.describe(include='all').transpose())

# Cardinalidad
print('\nCardinalidad:')
card = df.nunique(dropna=False).sort_values(ascending=False)
card


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
brand,1061,20,Bajaj,260,NaN,NaN,NaN,NaN,NaN,NaN,NaN
model,1061,276,Bajaj Pulsar 150,41,NaN,NaN,NaN,NaN,NaN,NaN,NaN
selling_price,1061.0,NaN,NaN,NaN,59638.151744,56304.291973,5000.0,28000.0,45000.0,70000.0,760000.0
year,1061.0,<NA>,<NA>,<NA>,2013.867107,4.301191,1988.0,2011.0,2015.0,2017.0,2020.0
seller_type,1061,2,Individual,1055,NaN,NaN,NaN,NaN,NaN,NaN,NaN
owner,1061,4,1st owner,924,NaN,NaN,NaN,NaN,NaN,NaN,NaN
km_driven,1061.0,NaN,NaN,NaN,34359.833176,51623.152702,350.0,13500.0,25000.0,43000.0,880000.0
ex_showroom_price,626.0,NaN,NaN,NaN,87958.714058,77496.587189,30490.0,54852.0,72752.5,87031.5,1278000.0



Cardinalidad:


km_driven            304
model                276
ex_showroom_price    231
selling_price        130
year                  28
brand                 20
owner                  4
seller_type            2
dtype: int64

### 7) Valores faltantes
Calcula el conteo y porcentaje de datos faltantes en cada fila

In [23]:
# Conteo y porcentaje de valores faltantes
na_counts = df.isna().sum().sort_values(ascending=False)
na_pct = (df.isna().mean() * 100).sort_values(ascending=False)

# Mostrar top 20 columnas con más faltantes
miss = pd.DataFrame({'na_count': na_counts, 'na_pct': na_pct})
display(miss.head(20))


,na_count,na_pct
brand,0,0.0
model,0,0.0
selling_price,0,0.0
year,0,0.0
seller_type,0,0.0
owner,0,0.0
km_driven,0,0.0
ex_showroom_price,0,0.0


### 8) Eliminación de duplicados
Se detectan y eliminan filas duplicadas exactas. Se reporta el nuevo Shape

In [24]:

dups = df.duplicated().sum()
print('Duplicados encontrados:', dups)
if dups > 0:
    df = df.drop_duplicates().reset_index(drop=True)
print('Shape tras quitar duplicados:', df.shape)


Duplicados encontrados: 6
Shape tras quitar duplicados: (1055, 8)


### 9) Eliminación de Outliers (IQR)
Se seleccionan las columnas numéricas, Calcula Q1 (percentil 25%) y Q3 (percentil 75%) y después Calcula el IQR (rango intercuartílico). Se acotan los valores extremos a [Q1-1.5*IQR, Q3+1.5*IQR].

In [25]:

num_cols = df.select_dtypes(include=['number']).columns
for c in num_cols:
    q1 = df[c].quantile(0.25)
    q3 = df[c].quantile(0.75)
    iqr = q3 - q1
    if pd.isna(iqr) or iqr == 0:
        continue
    low, high = q1 - 1.5*iqr, q3 + 1.5*iqr
    df[c] = df[c].clip(lower=low, upper=high)
df[num_cols].describe().transpose().head(10)


,count,mean,std,min,25%,50%,75%,max
selling_price,1055.0,54744.150711,35550.353741,5000.0,28000.0,45000.0,70000.0,133000.0
year,1055.0,2013.946919,4.022591,2002.0,2011.0,2015.0,2017.0,2020.0
km_driven,1055.0,30294.309005,22101.921628,350.0,13500.0,25000.0,43000.0,87250.0
ex_showroom_price,1055.0,72636.108531,13991.253723,43703.0,64589.0,72752.5,78513.0,99399.0


### 10) Codificación ligera y guardado
Cambio de categorías de baja cardinalidad a `category` y se guardan en un dataset limpio.

In [26]:

# Categorización ligera (<=50 categorías)
cat_candidates = [c for c in df.columns if df[c].dtype == 'object' and df[c].nunique() <= 50]
for c in cat_candidates:
    df[c] = df[c].astype('category')

# Guardamos
output_path = 'BikePrices_clean.csv'
df.to_csv(output_path, index=False)
print('Archivo guardado en:', output_path)
df.head(3)


Archivo guardado en: BikePrices_clean.csv


,brand,model,selling_price,year,seller_type,owner,km_driven,ex_showroom_price
0,TVS,TVS XL 100,30000,2017,Individual,1st owner,8000,43703.0
1,Bajaj,Bajaj ct 100,18000,2017,Individual,1st owner,35000,43703.0
2,Yo,Yo Style,20000,2011,Individual,1st owner,10000,43703.0
